In [22]:
import os
import glob
from pprint import pprint
from typing import List

#### Change the format of the ghostbuster data

In [3]:
root_dir = '/Users/xy/Github/ghostbuster-data'

# show folders
for folder in os.listdir(root_dir):
    if os.path.isdir(os.path.join(root_dir, folder)) and not folder.startswith('.'):
        print(folder)

reuter
perturb
other
wp
perturb_old
essay


In [36]:
def change_logprobs_format(logprobs_file) -> List:
    """
    Change the format of the logprobs file from column to a list of strings
    the original decimal (e.g. 7.8306713) is kept
    """
    result = []
    with open(logprobs_file, 'r') as f:
        for i, line in enumerate(f.readlines()):
            if line[-1] == '\n':
                line = line[:-1]
            if len(line) == 0:
                continue
            try:
                _, logprob = line.split(' ')
                result.append(logprob)
            except Exception as e:
                print(f'Error processing line {i} in {logprobs_file}: {e}')
                print(f'Line content: {line}')
                raise
    return result

In [ ]:
# test change_logprobs_format
logprob_file = '/Users/xy/Github/ghostbuster-data/essay/claude/logprobs/1-ada.txt'
res = change_logprobs_format(logprob_file)
print(len(res))
print(res[:10])

557
['7.8306713', '7.6627297', '7.753387', '1.7807125', '2.5496154', '1.1957519', '0.082388856', '2.8502815', '3.6934564', '4.9385443']


In [40]:
# essay

paren_dir = os.path.join(root_dir, 'essay')
folders_excluded = ['prompts']
child_dirs = [os.path.join(paren_dir, child) for child in os.listdir(paren_dir) 
                 if os.path.isdir(os.path.join(paren_dir, child)) and child not in folders_excluded]
# pprint(child_dirs)

if not os.path.exists('ghostbuster-data_reformed'):
    os.mkdir('ghostbuster-data_reformed')
output_dir = os.path.join('ghostbuster-data_reformed', 'essay')
if not os.path.exists(output_dir):
    os.mkdir(output_dir)


# process each logprob .txt file
for d in os.listdir(paren_dir)[:1]:
    if d in folders_excluded:
        continue
    child_dir = os.path.join(paren_dir, d)
    assert os.path.isdir(child_dir) and os.path.exists(child_dir)

    logprob_dir = os.path.join(child_dir, 'logprobs')
    assert os.path.isdir(logprob_dir) and os.path.exists(logprob_dir)

    # create output subfolder
    output_subdir = os.path.join(output_dir, d)
    if not os.path.exists(output_subdir):
        os.mkdir(output_subdir)

    logprob_files_ada = glob.glob(os.path.join(logprob_dir, '*-ada.txt'))
    logprobs_ada_reformed = []
    for lp_file in logprob_files_ada:
        try:
            res = change_logprobs_format(lp_file)
        except Exception as e:
            raise
        logprobs_ada_reformed.append(res)
        # save to individual output files
        output_file = os.path.join(output_subdir, os.path.basename(lp_file))
        with open(output_file, 'w') as f:
            f.write(' '.join(res))
    # save to combined output file
    output_file = os.path.join(output_subdir, 'combined-ada.txt')
    with open(output_file, 'w') as f:
        for res in logprobs_ada_reformed:
            f.write(' '.join(res) + '\n')

    logprob_files_davinci = glob.glob(os.path.join(logprob_dir, '*-davinci.txt'))
    logprobs_davinci_reformed = []
    for lp_file in logprob_files_davinci:
        try:
            res = change_logprobs_format(lp_file)
        except Exception as e:
            raise
        logprobs_davinci_reformed.append(res)
        # save to output file
        output_file = os.path.join(output_subdir, os.path.basename(lp_file))
        with open(output_file, 'w') as f:
            f.write(' '.join(res))
    # save to combined output file
    output_file = os.path.join(output_subdir, 'combined-davinci.txt')
    with open(output_file, 'w') as f:
        for res in logprobs_davinci_reformed:
            f.write(' '.join(res) + '\n')
